In [32]:
import os, sys, torch
import pandas as pd
import numpy as np

root = os.path.abspath('../../modules')
sys.path.append(root)
sys.path.append(os.path.join(root, 'mnist'))
sys.path.append(os.path.join(root, 'celeba_male'))
sys.path.append(os.path.join(root, 'imagenet'))

import mnist.batch as bt_mnist
import celeba_male.batch as bt_celeba
import imagenet.batch as bt_imagenet

num_experiments = 10

thresholds = {"MNIST": 0.02, "CelebA": 0.02, "ImageNet-1K": 0.18}

algorithms = {"MNIST": ["Gradient Ascent (A)", "Ascent descent (A-D)", "Gradient surgery (S)",\
                        "UNO", "UNO-S", r"\cite{moon2024feature}"],
             "CelebA": ["Gradient surgery (S)",\
                        "UNO", "UNO-S", r"\cite{moon2024feature}"],
             "ImageNet-1K": ["Gradient surgery (S)",\
                        "UNO", "UNO-S", r"ESD-u-1~\cite{gandikota2023erasing}",\
                             r"ESD-u-3~\cite{gandikota2023erasing}"]}

folders = {"MNIST": ["vae-a", "vae-ad", "vae-s",\
                     "vae-o", "vae-os", "vae-fu"],
             "CelebA": ["vae-s", "vae-o", "vae-os", "vae-fu"],
             "ImageNet-1K": ["dit-s", "dit-o", "dit-os", "dit-esd-1", "dit-esd-3"]}


roots = {"MNIST": "../../data/MNIST/MNIST-Experiments",\
         "CelebA": "../../data/CelebA/CelebA-Experiments",\
         "ImageNet-1K": "../../data/ImageNet/ImageNet-Experiments"}

paths = {}

for dataset, root in roots.items():
    # print(dataset, root)
    paths[dataset] = []
    paths[dataset] += [root + f'/{f}' for f in folders[dataset]]



params = {"MNIST": {"model": f"{roots["MNIST"]}/vae/vae_200.pth",
                    "latent_dim": 2,
                    "num_steps": 50,
                    "batch_size": 128,
                    "log_interval": 1,
                    "collect_interval": "epoch",
                    "save_steps": "epoch",
                    "all_digits": list(range(10)),
                    "forget_digit": 1,
                    "classifier_path": f"{roots["MNIST"]}/classifiers/MNISTClassifier.pth",
                    "kl_weight": 1,
                    "total_duration": None,
                    "data_path": roots["MNIST"]},
          
            "CelebA": {"model": f"{roots["CelebA"]}/vae/vae_200.pth",
                        "latent_dim": 512,
                        "num_steps": 50,
                        "batch_size": 128,
                        "log_interval": 1,
                        "collect_interval": "epoch",
                        "save_steps": "epoch",
                        "all_classes": [0, 1],
                        "forget_class": 1,
                        "classifier_path": f"{roots["CelebA"]}/cnn/cnn_10.pth",
                        "kl_weight": 1,
                        "total_duration": None,
                        "data_path": f'{roots["CelebA"]}/dataset',
                        "max_data": None},
          
            "ImageNet-1K": {
                        "model_path": f"{roots["ImageNet-1K"]}/DiT-XL-2",
                        "num_steps": 100,
                        "batch_size": 10,
                        "log_interval": 1,
                        "collect_interval": "epoch",
                        "save_steps": None,
                        "exchange_classes": [208],
                        "forget_class": 207,
                        "data_path": "../../data/ImageNet/2012",
                        "imagenet_json_path": f"{roots["ImageNet-1K"]}/imagenet_class_index.json",
                        "freeze_K": 4,
                        "unfreeze_last": True,
                        "keep_all": True,
                        "n_samples": 50,
                        "device": "mps",
                        "diffusion_steps": 15,
                        "learning_rate": 1e-4,}
}

modules = {"MNIST": bt_mnist, "CelebA": bt_celeba, "ImageNet-1K": bt_imagenet}

batches = {}

for dataset, path in paths.items():
    batches[dataset] = []
    batches[dataset] += [modules[dataset].BatchExperiment\
                           (None, params[dataset] | {"folder": path}, num_experiments)\
                         for path in paths[dataset]]


summary = {}

for dataset, path in batches.items():
    summary[dataset] = dict(zip(algorithms[dataset], [b.summarize_wo_fid(thresholds[dataset])\
                                                      for b in batches[dataset]]))

metadata = {"MNIST": r"""\begingroup
\setlength{\tabcolsep}{1.8pt} 
\begin{table}[htp]
\caption{Performance of various algorithms for class/feature unlearning with VAEs on MNIST and CelebA and diffusion transformers on ImageNet-1K. Each experiment is repeated $10$ times, and the standard deviations are shown in parentheses. FID values are calculated using $25,000$ samples for MNIST and CelebA and $22,000$ samples for ImageNet-1K. {\color{red}\ding{55}} indicates that the generated samples after unlearning are unrecognizably different from the original model. Bold indicates the best score among instances with acceptable FID. {\color{green!60!black}\ding{51}} indicates the generated samples after unlearning are perceptually indistinguishable from the original model in terms of visual fidelity. An asterisk (*) denotes cases where the algorithm failed to reach the target fraction of forget samples in the generated images within the allotted training steps. We only report on (A) and (A-D) for MNIST since they demonstrate catastrophic forgetting in all cases; for CelebA (A) and (A-D) result in NaN for FID, generating white images. The original models for MNIST, CelebA and ImageNet-1K have FID $20.7$, $166.3$ and $12.7$ respectively, and we are unlearning the digit $1$, faces identified as "male" and images of class $207$ (golden retriever) respectively.}
{\scriptsize
\label{tab:res}
\vspace{0.5em}
\centering
\begin{tabular}{@{}ll@{\hspace{-1em}}c@{\hspace{-0.5em}}c@{\hspace{-0.2em}}c@{\hspace{-0.5em}}c@{}}
\toprule
\textbf{Dataset} & \textbf{Algorithm} & \hspace{-1em}Time to unlearn (s) ↓& FID ↓& $\Delta t$ (ms) & $5$-step f-ratio ↓\\
\midrule
\multirow{5}{*}{\shortstack{MNIST }}""",

"CelebA": r"""\midrule
\multirow{3}{*}{\shortstack{CelebA}}""",

            
"ImageNet-1K": r"""\midrule
\multirow{3}{*}{\shortstack{ImageNet-1K}}""",

"end": r"""\bottomrule
\end{tabular}
\vspace{1.5em}}
\end{table}""",
           }

In [33]:
from decimal import Decimal, ROUND_HALF_UP

def r(x, n):
    return format(
        Decimal(str(x)).quantize(Decimal("1." + "0"*n), rounding=ROUND_HALF_UP),
        f".{n}f"
    )

def bold(s, flag):
    return rf"\textbf{{{s}}}" if flag else s

def row(algo, mean, std, best, fid_ref=250, time_star_threshold=None):
    t_mean, t_std = mean["Time"], std["Time"]
    fid_mean, fid_std = mean["FID"], std["FID"]
    ms_mean, ms_std = 1000 * mean["Time/Step"], 1000 * std["Time/Step"]
    frac_mean, frac_std = mean["5-step remainder"], std["5-step remainder"]

    fid_ok = fid_mean < fid_ref

    time_str = bold(r(t_mean, 3), algo in best["Time"])
    fid_str  = bold(r(fid_mean, 1), algo in best["FID"])
    ms_str   = r(ms_mean, 1)
    frac_str = bold(r(frac_mean, 2), algo in best["5-step remainder"])

    star = r"$^*$" if (time_star_threshold is not None and t_mean > time_star_threshold) else ""
    mark = r" {\color{green!60!black}\ding{51}}" if fid_ok else r" {\color{red}\ding{55}}"

    return (
        f"  & {algo}"
        f" & {time_str}{star} ({r(t_std,3)})"
        f" & {fid_str} ({r(fid_std,1)}){mark}"
        f" & {ms_str} ({r(ms_std,2)})"
        f" & {frac_str} ({r(frac_std,2)})\\\\"
    )

def best_algorithms(rows, fid_ref):
    vals = {}
    for algo, (mean, std) in rows.items():
        vals[algo] = {
            "Time": mean["Time"],
            "FID": mean["FID"],
            "5-step remainder": mean["5-step remainder"],
        }

    best = {}

    for key in ["Time", "5-step remainder"]:
        m = min(v[key] for v in vals.values())
        best[key] = {algo for algo, v in vals.items() if v[key] == m}

    valid = {algo: v["FID"] for algo, v in vals.items() if v["FID"] < fid_ref}
    if valid:
        m = min(valid.values())
        best["FID"] = {algo for algo, x in valid.items() if x == m}
    else:
        best["FID"] = set()

    return best

def block(metadata, summary, dataset, fid_ref=250, time_star_threshold=None):
    rows = summary[dataset]
    best = best_algorithms(rows, fid_ref)

    out = [metadata[dataset]]
    for algo, (mean, std) in rows.items():
        out.append(
            row(
                algo,
                mean,
                std,
                best=best,
                fid_ref=fid_ref,
                time_star_threshold=time_star_threshold,
            )
        )
    return "\n".join(out)

def table(metadata, summary, fid_ref=250, time_star_threshold=None):
    out = []
    for dataset in summary:
        out.append(block(metadata, summary, dataset, fid_ref, time_star_threshold))
    out.append(metadata["end"])
    return "\n".join(out)

print(table(metadata, summary, fid_ref=250, time_star_threshold=10))

\begingroup
\setlength{\tabcolsep}{1.8pt} 
\begin{table}[htp]
\caption{Performance of various algorithms for class/feature unlearning with VAEs on MNIST and CelebA and diffusion transformers on ImageNet-1K. Each experiment is repeated $10$ times, and the standard deviations are shown in parentheses. FID values are calculated using $25,000$ samples for MNIST and CelebA and $22,000$ samples for ImageNet-1K. {\color{red}\ding{55}} indicates that the generated samples after unlearning are unrecognizably different from the original model. Bold indicates the best score among instances with acceptable FID. {\color{green!60!black}\ding{51}} indicates the generated samples after unlearning are perceptually indistinguishable from the original model in terms of visual fidelity. An asterisk (*) denotes cases where the algorithm failed to reach the target fraction of forget samples in the generated images within the allotted training steps. We only report on (A) and (A-D) for MNIST since they demon

In [34]:
summary["ImageNet-1K"][r"ESD-3~\cite{gandikota2023erasing}"]

KeyError: 'ESD-3~\\cite{gandikota2023erasing}'